In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import patheffects
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from pathlib import Path

import importlib
import track

# Development helper: uncomment after editing track.py.
# importlib.reload(track)

# Load Snapshot

按区域窗口加载单日 OFES NP30 快照。MOM3 Arakawa B-grid 的 u/v 均由四角共置到示踪物中心；pressure 保留为 dbar-like 原生坐标，w 保留在层界面。

In [ ]:
snap = track.load_ofes_snapshot(
    '2003-04-05', lon_bounds=(142, 150), lat_bounds=(32, 39),
    pressure_bounds=(0, 1100),
)
print(list(snap.keys()))
print(f"do2: {snap['do2'].shape}, u: {snap['u'].shape}, w: {snap['w'].shape}")

# Quick-Look Horizontal Slice

In [ ]:
track.plot_ofes_snapshot_quick(snap, variable='do2', pressure=600.0)

# Vertical Profile Extraction

从快照中双线性插值提取定压虚拟剖面；Pressure / do2 / temp / salinity 继续送入通用单剖面 detector。

In [ ]:
prof = track.extract_ofes_profile_interp(snap, lon=144.5, lat=35.0,
                                         variables=['do2', 'temp', 'salinity'])
prof.head(10)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 6), sharey=True)
for ax, var, label in zip(axes, ['do2', 'temp', 'salinity'],
                          ['DO₂ (μmol kg⁻¹)', 'Potential temp (°C)', 'Salinity (PSS-78)']):
    ax.plot(prof[var], prof['Pressure'], linewidth=1.2)
    ax.set_xlabel(label)
    ax.invert_yaxis()
axes[0].set_ylabel('Pressure (dbar)')
fig.suptitle(f"OFES profile  144.5°E, 35°N  {snap['date'].strftime('%Y-%m-%d')}")
fig.tight_layout()

# δDO Detection on OFES

复用观测 pipeline 的 `calculate_delta_do` 在 OFES 虚拟剖面上检测异常。

In [ ]:
det_cfg = track.make_detection_config('do')

result = track.detect_ofes_delta_do(snap, lon=144.5, lat=35.0,
                                    detection_config=det_cfg)
result

In [ ]:
det_prof = track.extract_ofes_profile_interp(snap, lon=144.5, lat=35.0,
                                             variables=['do2'])
fig, ax = plt.subplots(figsize=(5, 7))
ax.plot(det_prof['do2'], det_prof['Pressure'], 'b-', linewidth=1.2)
ax.axhline(det_cfg.anomaly_min_depth, color='gray', ls='--', alpha=0.5,
           label=f'min pressure {det_cfg.anomaly_min_depth:.0f} dbar')
for _, row in result.iterrows():
    ax.plot(row['do_value'], row['pressure'], 'ro', ms=7)
    ax.annotate(f"ΔDO={row['delta_do']:.1f}", (row['do_value'], row['pressure']),
                textcoords='offset points', xytext=(8, 0), color='red', fontsize=9)
ax.invert_yaxis()
ax.set_xlabel('DO₂ (μmol kg⁻¹)')
ax.set_ylabel('Pressure (dbar)')
ax.set_title(f"δDO detection  144.5°E, 35°N  {snap['date'].strftime('%Y-%m-%d')}")
ax.legend(loc='lower left')
fig.tight_layout()

# Pressure-Surface Advection

固定 pressure 的二维 RK4 诊断。日场时间插值与三维 w 运动仍未验证，结果只用于检查水平输运代码和粒子状态。

In [ ]:
snap_prev = track.load_ofes_snapshot(
    '2003-04-04', variables=['u', 'v'], lon_bounds=(142, 150),
    lat_bounds=(32, 39), pressure_bounds=(300, 700),
)

# 从检测到的异常点与同一 pressure 周围对照点释放粒子。
anom_pressure = result.iloc[0]['pressure']
anom_lat = result.iloc[0]['Latitude']
anom_lon = result.iloc[0]['Longitude']

# 异常点 + 沿纬度或经度 ±1°/±2° 共 9 个点。
offsets = [
    (0, 0),
    (0, -2), (0, -1), (0, +1), (0, +2),
    (-2, 0), (-1, 0), (+1, 0), (+2, 0),
]
particles = np.array([[anom_pressure, anom_lat + dlat, anom_lon + dlon]
                       for dlat, dlon in offsets])
labels = [f"({dlat:+d},{dlon:+d})" if (dlat, dlon) != (0, 0) else "anomaly"
          for dlat, dlon in offsets]

traj_bwd = track.advect_ofes_particles([snap_prev, snap], particles, backward=True)
traj_pos = traj_bwd['positions']

print(f"Release pressure: {anom_pressure:.0f} dbar | backward 1 day")
for i, lbl in enumerate(labels):
    d = traj_pos[-1, i] - traj_pos[0, i]
    km = np.sqrt((d[1]*111.32)**2 + (d[2]*111.32*np.cos(np.radians(particles[i,1])))**2)
    print(f"  {lbl:>12s}: status={traj_bwd['final_status'][i]:>7s} "
          f"Δlat={d[1]:+.4f}° Δlon={d[2]:+.4f}° Δp={d[0]:+.1f} dbar  ~{km:.0f} km")

In [ ]:
bc = track._BASEMAP_COLORS
fig, ax = plt.subplots(figsize=(10, 7), subplot_kw={'projection': ccrs.PlateCarree()})
k = int(np.argmin(np.abs(snap['pressure'] - anom_pressure)))
im = ax.pcolormesh(snap['lon'], snap['lat'], snap['do2'][k],
                   transform=ccrs.PlateCarree(), cmap='viridis', shading='auto')
ax.add_feature(cfeature.LAND, facecolor=bc['land'], edgecolor=bc['coastline'],
               linewidth=0.5, zorder=2)
gl = ax.gridlines(draw_labels=True, linewidth=0.3, color=bc['grid'], alpha=0.6)
gl.top_labels = False
gl.right_labels = False
fig.colorbar(im, ax=ax, shrink=0.7, label='DO₂ (μmol kg⁻¹)')

p0, p1 = traj_bwd['positions'][0], traj_bwd['positions'][-1]
for i in range(len(particles)):
    color = 'red' if labels[i] == 'anomaly' else 'white'
    ms = 8 if labels[i] == 'anomaly' else 5
    ax.plot(p0[i, 2], p0[i, 1], 'o', color=color, ms=ms,
            markeredgecolor='black', markeredgewidth=0.5,
            transform=ccrs.PlateCarree(), zorder=4)
    ax.annotate('', xy=(p1[i, 2], p1[i, 1]), xytext=(p0[i, 2], p0[i, 1]),
                arrowprops=dict(arrowstyle='->', color=color, lw=1.5,
                                path_effects=[patheffects.withStroke(linewidth=2.5, foreground='black')]),
                transform=ccrs.PlateCarree())

ax.plot([], [], 'ro', ms=8, markeredgecolor='black', label='anomaly')
ax.plot([], [], 'o', color='white', ms=5, markeredgecolor='black', label='control')
ax.plot([], [], '->', color='gray', label='backward 1d')
ax.legend(loc='lower left')

ax.set_title(f"Fixed-pressure backward diagnostic ({anom_pressure:.0f} dbar)  {snap['date'].strftime('%Y-%m-%d')}")
fig.tight_layout()